# 00 - Data Audit

Verify all data sources merged correctly and describe the panel.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

PANEL_FILE = "analysis_panel.parquet"
if not (DATA_DIR / PANEL_FILE).exists():
    raise FileNotFoundError(
        "Analysis panel not found. Build it by running in order:\n"
        "  python scripts/download_epa_aqs.py --email EMAIL --key KEY\n"
        "  python scripts/download_hms_smoke.py\n"
        "  python scripts/download_seda.py  (manual — see instructions)\n"
        "  python src/merge/build_crosswalks.py\n"
        "  python src/ingest/epa_aqs.py\n"
        "  python src/ingest/seda.py\n"
        "  python src/exposure/smoke_instrument.py\n"
        "  python src/merge/build_panel.py"
    )

panel = pd.read_parquet(DATA_DIR / PANEL_FILE)
print(f"Panel: {panel.shape}")
print(f"Districts: {panel['leaid'].nunique()}")
print(f"Years: {sorted(panel['year'].dropna().unique())}")

## Panel completeness

In [ ]:
print("Missing values by column:")
print(panel.isnull().sum()[panel.isnull().sum() > 0])
print(f"\nRows with complete PM2.5 + smoke + test score: {panel[['pm25_annual_mean','smoke_days','test_score_mean']].dropna().shape[0]:,}")

## Key variable distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
panel["pm25_annual_mean"].dropna().hist(bins=40, ax=axes[0], color="steelblue", alpha=0.75)
axes[0].set_title("PM2.5 Annual Mean (μg/m³)")
panel["smoke_days"].dropna().hist(bins=40, ax=axes[1], color="orange", alpha=0.75)
axes[1].set_title("Smoke Days per Year")
panel["test_score_mean"].dropna().hist(bins=40, ax=axes[2], color="green", alpha=0.75)
axes[2].set_title("Test Score (standardized)")
plt.tight_layout()
plt.savefig(OUT_DIR / "00_distributions.png", bbox_inches="tight")
plt.show()

## PM2.5 vs smoke instrument correlation (first-stage preview)

In [ ]:
corr = panel[["pm25_annual_mean","smoke_days"]].corr().iloc[0,1]
print(f"corr(PM2.5, smoke_days) = {corr:.3f}")
print("Strong correlation needed for IV relevance (first-stage F > 10)")